In [1]:
import tmap as tm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import rdkit
from rdkit import Chem
from rdkit.Chem import MACCSkeys
from rdkit.Chem import Draw
from bokeh.plotting import figure, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.io import export_png
import base64
from io import BytesIO

# Análise de Espaço Químico com TMAP

Este notebook gera visualizações de TMAP (Tree MAP) para análise do espaço químico dos datasets de DeNovo HsDHODH.

## Estrutura do Notebook:

1. **Imports** - Bibliotecas necessárias (tmap, rdkit, matplotlib, bokeh)
2. **Carregamento de Dados** - Leitura dos datasets druglike e highdiv
3. **Funções de Fingerprints** - MACCS e ECFP4
4. **Função create_tmap_data** - Calcula o TMAP para um dataset
5. **Cálculo dos TMAPs** - Processa todos os datasets e armazena resultados
6. **Gráficos Estáticos** - Gera visualizações PNG com matplotlib
7. **Gráficos Interativos** - Gera visualizações HTML com Bokeh (hover mostra molécula e dataset)

## Datasets Processados:
- **Diversidade**: DRUGLIKE, HIGHDIV
- **Descritores**: MACCS, ECFP4
- **Datasets**: CONCAT, CRAFT, LANA, MAY

**Total**: 16 visualizações (8 PNG + 8 HTML)

In [11]:
df_druglike = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Datasets/concat_datasets_druglike.csv')
df_highdiv = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Datasets/concat_datasets_highdiv.csv')

In [12]:
df_druglike

,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical,SA_Score,SA_Score_Binary,Dataset
0,OC(=O)[C@H](/C=C/Cc1nc(ccc1)CCCl)c1nc(N)c2c(n1...,21/000000020,InChI=1S/C17H17ClN6O2/c18-8-7-11-4-1-3-10(21-1...,C17H17N6O2Cl,372,2.17,6.45,6.97,24.10,-190,3.895024,1,DRUGLIKE_CONCAT
1,OC(=O)[C@H](/C=C/Cc1ncccc1CCBr)c1nc(N)c2c(n1)[...,21/000000030,InChI=1S/C17H17BrN6O2/c18-7-6-10-3-2-8-20-13(1...,C17H17N6O2Br,417,2.42,6.04,5.01,24.10,-190,3.978020,1,DRUGLIKE_CONCAT
2,Oc1ncc(c(n1)C/C=C/c1nc(Br)c(cc1)Cl)C(=O)O\tNo_...,29/000000167,InChI=1S/C13H9BrClN3O3/c14-11-9(15)5-4-7(17-11...,C13H9N3O3ClBr,370,2.46,5.37,7.30,5.21,0,3.366448,1,DRUGLIKE_CONCAT
3,OC(=O)Cc1cccc(c1)c1[nH]c(cc1)C(=O)c1nc(=O)o[nH...,23/000000136,InChI=1S/C15H11N3O5/c19-12(20)7-8-2-1-3-9(6-8)...,C15H11N3O5,313,1.20,5.37,6.90,3.58,-120,2.854078,1,DRUGLIKE_CONCAT
4,OCCc1cccc(c1)c1[nH]c(cc1)C(=O)c1nc(=O)o[nH]1\t...,23/000000238,InChI=1S/C15H13N3O4/c19-7-6-9-2-1-3-10(8-9)11-...,C15H13N3O4,299,1.12,5.35,6.43,3.58,-120,2.906767,1,DRUGLIKE_CONCAT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1687,c1nc2c(nc1CN)[nH]cc2C(=C)C(=O)OCCF\tNo_3_188,28/000000005,InChI=1S/C12H13FN4O2/c1-7(12(18)19-3-2-13)9-6-...,C12H13N4O2F,264,0.61,4.00,1.03,19.70,-60,3.366012,1,DRUGLIKE_MAY
1688,OC(=O)c1nnc(o1)c1cc(=O)cc([nH]1)CCC=C\tNo_2_464,15/000000082,InChI=1S/C12H11N3O4/c1-2-3-4-7-5-8(16)6-9(13-7...,C12H11N3O4,261,1.78,4.00,3.40,24.80,-120,3.224311,1,DRUGLIKE_MAY
1689,Oc1noc(c1C(=O)O)/C=C/C(=O)OCC\tNo_3_585,7/000000002,InChI=1S/C9H9NO6/c1-2-15-6(11)4-3-5-7(9(13)14)...,C9H8NO6,226,0.40,4.00,4.28,2.35,-20,2.907700,1,DRUGLIKE_MAY
1690,OC(=O)[C@@H](c1cccc(c1)O)N\tNo_2_4,10/000000006,InChI=1S/C8H9NO3/c9-7(8(11)12)5-2-1-3-6(10)4-5...,C8H9NO3,167,-0.01,4.00,2.32,1.00,-150,2.449543,1,DRUGLIKE_MAY


In [8]:
def smiles_to_maccs(smiles_list):
    return [MACCSkeys.GenMACCSKeys(mol) if (mol := Chem.MolFromSmiles(smiles)) else None for smiles in smiles_list]

def smiles_to_ECFP4(smiles_list):
    return [Chem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) if (mol := Chem.MolFromSmiles(smiles)) else None for smiles in smiles_list]


,"SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical,SA_Score,SA_Score_Binary,Dataset"
OC(=O)[C@H](/C=C/Cc1nc(ccc1)CCCl)c1nc(N)c2c(n1)[nH]nc2,"No_3_262,21/000000020,""InChI=1S/C17H17ClN6O2/c..."
OC(=O)[C@H](/C=C/Cc1ncccc1CCBr)c1nc(N)c2c(n1)[nH]nc2,"No_4_305,21/000000030,""InChI=1S/C17H17BrN6O2/c..."
Oc1ncc(c(n1)C/C=C/c1nc(Br)c(cc1)Cl)C(=O)O,"No_2_469,29/000000167,""InChI=1S/C13H9BrClN3O3/..."
OC(=O)Cc1cccc(c1)c1[nH]c(cc1)C(=O)c1nc(=O)o[nH]1,"No_4_112,23/000000136,""InChI=1S/C15H11N3O5/c19..."
OCCc1cccc(c1)c1[nH]c(cc1)C(=O)c1nc(=O)o[nH]1,"No_4_625,23/000000238,""InChI=1S/C15H13N3O4/c19..."
...,...
c1nc2c(nc1CN)[nH]cc2C(=C)C(=O)OCCF,"No_3_188,28/000000005,""InChI=1S/C12H13FN4O2/c1..."
OC(=O)c1nnc(o1)c1cc(=O)cc([nH]1)CCC=C,"No_2_464,15/000000082,""InChI=1S/C12H11N3O4/c1-..."
Oc1noc(c1C(=O)O)/C=C/C(=O)OCC,"No_3_585,7/000000002,""InChI=1S/C9H9NO6/c1-2-15..."
OC(=O)[C@@H](c1cccc(c1)O)N,"No_2_4,10/000000006,""InChI=1S/C8H9NO3/c9-7(8(1..."


In [4]:
def create_tmap_data(df, fingerprint_type='maccs'):
    """
    Cria os dados necessários para gerar um TMAP
    
    Args:
        df: DataFrame com coluna 'SMILES'
        fingerprint_type: 'maccs' ou 'ecfp4'
    
    Returns:
        layout: dados do layout do TMAP
        df: DataFrame atualizado com fingerprints
    """
    # Gerar fingerprints
    if fingerprint_type == 'maccs':
        df['fingerprints'] = [smiles_to_maccs(smiles) for smiles in df['SMILES']]
        n_bits = 166  # MACCS tem 166 bits
    elif fingerprint_type == 'ecfp4':
        df['fingerprints'] = [smiles_to_ECFP4(smiles) for smiles in df['SMILES']]
        n_bits = 2048  # ECFP4 tem 2048 bits
    else:
        raise ValueError("fingerprint_type deve ser 'maccs' ou 'ecfp4'")
    
    # Remover linhas sem fingerprints
    df = df.dropna(subset=['fingerprints'])
    
    # Criar MinHash
    minhashes = []
    minhash = tm.Minhash(n_bits)
    for fp in df['fingerprints']:
        minhashes.append(minhash.from_binary_array(list(fp)))
    
    # Criar LSH Forest
    lf = tm.LSHForest(n_bits, 128)
    lf.batch_add(minhashes)
    lf.index()
    
    # Criar grafo k-NN
    k = 10
    knng_from = tm.VectorUint()
    knng_to = tm.VectorUint()
    knng_weight = tm.VectorFloat()
    lf.get_knn_graph(knng_from, knng_to, knng_weight, k)
    
    # Criar layout
    edges_list = [(knng_from[i], knng_to[i], knng_weight[i]) for i in range(len(knng_from))]
    num_vertices = len(df)
    layout = tm.layout_from_edge_list(num_vertices, edges_list, create_mst=True)
    
    return layout, df


In [ ]:
# Criar diretório de resultados se não existir
import os
output_dir = '/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Results/Chemical_Spaces/TMAP'
os.makedirs(output_dir, exist_ok=True)

# Dicionário para armazenar os resultados dos TMAPs calculados
tmap_results = {}

# Gerar TMAPs individuais para cada dataset e descritor
datasets_info = [
    ('DRUGLIKE', df_druglike),
    ('HIGHDIV', df_highdiv)
]

fingerprint_types = ['MACCS', 'ECFP4']
dataset_names = ['CONCAT', 'CRAFT', 'LANA', 'MAY']

print("="*60)
print("CALCULANDO TMAPS")
print("="*60)

for diversity_type, df_full in datasets_info:
    print(f"\n{'='*60}")
    print(f"Processando: {diversity_type}")
    print('='*60)
    
    for dataset_name in dataset_names:
        # Filtrar o dataset específico
        df_subset = df_full[df_full['Dataset'].str.contains(dataset_name, case=False, na=False)].copy()
        
        if len(df_subset) == 0:
            print(f"⚠️  {dataset_name}: Nenhum dado encontrado")
            continue
            
        print(f"\n{dataset_name}: {len(df_subset)} moléculas")
        
        for descriptor in fingerprint_types:
            print(f"  - Calculando TMAP com {descriptor}...", end=' ')
            
            try:
                # Criar TMAP
                layout, df_processed = create_tmap_data(df_subset, fingerprint_type=descriptor.lower())
                
                # Armazenar resultados
                key = f"{diversity_type}_{descriptor}_{dataset_name}"
                tmap_results[key] = {
                    'layout': layout,
                    'df': df_processed,
                    'diversity': diversity_type,
                    'descriptor': descriptor,
                    'dataset': dataset_name
                }
                
                print(f"✓ Completo ({len(df_processed)} moléculas)")
                
            except Exception as e:
                print(f"✗ Erro: {str(e)}")

print(f"\n{'='*60}")
print(f"Cálculo completo! {len(tmap_results)} TMAPs calculados.")
print('='*60)

CALCULANDO TMAPS

Processando: DRUGLIKE

CONCAT: 843 moléculas
  - Calculando TMAP com MACCS... 

CALCULANDO TMAPS

Processando: DRUGLIKE

CONCAT: 843 moléculas
  - Calculando TMAP com MACCS... 

RDKit ERROR: [13:34:16] SMILES Parse Error: syntax error while parsing: (
[13:34:16] SMILES Parse Error: syntax error while parsing: (
RDKit ERROR: [13:34:16] SMILES Parse Error: Failed parsing SMILES '(' for input: '('
RDKit ERROR: [13:34:16] SMILES Parse Error: syntax error while parsing: =
RDKit ERROR: [13:34:16] SMILES Parse Error: Failed parsing SMILES '=' for input: '='
RDKit ERROR: [13:34:16] SMILES Parse Error: syntax error while parsing: )
RDKit ERROR: [13:34:16] SMILES Parse Error: Failed parsing SMILES ')' for input: ')'
RDKit ERROR: [13:34:16] SMILES Parse Error: syntax error while parsing: [
RDKit ERROR: [13:34:16] SMILES Parse Error: Failed parsing SMILES '[' for input: '['
RDKit ERROR: [13:34:16] SMILES Parse Error: syntax error while parsing: @
RDKit ERROR: [13:34:16] SMILES Parse Error: Failed parsing SMILES '@' for input: '@'
RDKit ERROR: [13:34:16] SMILES Parse Error: syntax error while parsing: H
RDKit ERROR: [13:34:16] SMILES Parse Error: Failed parsing SMILES 'H' 

In [ ]:
# Gerar gráficos estáticos com Matplotlib
print("="*60)
print("GERANDO GRÁFICOS ESTÁTICOS (PNG)")
print("="*60)

for key, data in tmap_results.items():
    layout = data['layout']
    df_processed = data['df']
    diversity_type = data['diversity']
    descriptor = data['descriptor']
    dataset_name = data['dataset']
    
    print(f"Plotando: {key}...", end=' ')
    
    try:
        # Extrair coordenadas
        x, y = zip(*[(layout[0][i], layout[1][i]) for i in range(len(df_processed))])
        
        # Criar visualização
        plt.figure(figsize=(10, 8))
        plt.scatter(x, y, s=5, alpha=0.6, c='steelblue')
        plt.xlabel("Dimensão 1", fontsize=12)
        plt.ylabel("Dimensão 2", fontsize=12)
        plt.title(f"TMAP - {diversity_type} - {dataset_name} - {descriptor}\n({len(df_processed)} moléculas)", 
                 fontsize=14, fontweight='bold')
        
        # Salvar figura com formato: DIVERSITY_DESCRIPTOR_DATASET.png
        filename = f"{diversity_type}_{descriptor}_{dataset_name}.png"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"✓ Salvo")
        
    except Exception as e:
        print(f"✗ Erro: {str(e)}")

print(f"\n{'='*60}")
print(f"Gráficos estáticos salvos em: {output_dir}")
print('='*60)

In [ ]:
# Função para converter molécula em imagem base64
def mol_to_base64(smiles, size=(150, 150)):
    """Converte SMILES em imagem PNG codificada em base64"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return ""
        img = Draw.MolToImage(mol, size=size)
        buffered = BytesIO()
        img.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue()).decode()
        return f"data:image/png;base64,{img_str}"
    except:
        return ""

# Gerar gráficos interativos com Bokeh
print("="*60)
print("GERANDO GRÁFICOS INTERATIVOS (HTML)")
print("="*60)

for key, data in tmap_results.items():
    layout = data['layout']
    df_processed = data['df']
    diversity_type = data['diversity']
    descriptor = data['descriptor']
    dataset_name = data['dataset']
    
    print(f"Plotando: {key}...", end=' ')
    
    try:
        # Extrair coordenadas
        x = [layout[0][i] for i in range(len(df_processed))]
        y = [layout[1][i] for i in range(len(df_processed))]
        
        # Preparar dados para Bokeh
        df_plot = df_processed.copy().reset_index(drop=True)
        df_plot['x'] = x
        df_plot['y'] = y
        
        # Gerar imagens das moléculas
        print("(gerando imagens...", end=' ')
        df_plot['mol_image'] = df_plot['SMILES'].apply(mol_to_base64)
        print("OK)", end=' ')
        
        # Criar ColumnDataSource
        source = ColumnDataSource(df_plot)
        
        # Criar figura
        p = figure(
            width=900,
            height=700,
            title=f"TMAP - {diversity_type} - {dataset_name} - {descriptor} ({len(df_plot)} moléculas)",
            tools="pan,wheel_zoom,box_zoom,reset,save",
            toolbar_location="right"
        )
        
        # Adicionar pontos
        circles = p.circle(
            'x', 'y',
            size=5,
            alpha=0.6,
            color='steelblue',
            source=source
        )
        
        # Configurar HoverTool com imagem da molécula
        hover = HoverTool(
            tooltips="""
            <div style="width:200px;">
                <div>
                    <img src="@mol_image" style="width:150px; height:150px; border:1px solid #ddd; padding:5px;">
                </div>
                <div style="margin-top:5px;">
                    <span style="font-weight:bold;">Dataset:</span> @Dataset<br>
                    <span style="font-weight:bold;">SMILES:</span> @SMILES<br>
                    <span style="font-weight:bold;">MW:</span> @MW<br>
                    <span style="font-weight:bold;">LogP:</span> @LogP
                </div>
            </div>
            """,
            renderers=[circles]
        )
        p.add_tools(hover)
        
        # Configurar labels dos eixos
        p.xaxis.axis_label = "Dimensão 1"
        p.yaxis.axis_label = "Dimensão 2"
        p.title.text_font_size = "14pt"
        
        # Salvar como HTML
        filename = f"{diversity_type}_{descriptor}_{dataset_name}.html"
        filepath = os.path.join(output_dir, filename)
        output_file(filepath)
        save(p)
        
        print(f"✓ Salvo")
        
    except Exception as e:
        print(f"✗ Erro: {str(e)}")

print(f"\n{'='*60}")
print(f"Gráficos interativos salvos em: {output_dir}")
print('='*60)

## Instruções de Uso

### Instalação de Dependências (se necessário):
```bash
pip install tmap bokeh rdkit
```

### Como Executar:

1. **Execute as células 1-4** para carregar bibliotecas e dados
2. **Execute a célula 5** para calcular todos os TMAPs (pode demorar alguns minutos)
3. **Execute a célula 6** para gerar gráficos estáticos PNG
4. **Execute a célula 7** para gerar gráficos interativos HTML

### Saídas:

Todos os arquivos são salvos em:  
`/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Results/Chemical_Spaces/TMAP`

**Formato de nomes:**
- PNG: `DIVERSITY_DESCRIPTOR_DATASET.png`
- HTML: `DIVERSITY_DESCRIPTOR_DATASET.html`

### Gráficos Interativos:

Abra os arquivos HTML no navegador. Ao passar o mouse sobre cada ponto:
- **Mostra**: Imagem da molécula
- **Informações**: Dataset, SMILES, MW, LogP